In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Carregar os dados
train_path = "C:\\Users\\felpp\\Downloads\\titanic\\train.csv"
test_path = "C:\\Users\\felpp\\Downloads\\titanic\\test.csv"
gabarito_path = "C:\\Users\\felpp\\Downloads\\titanic\\gender_submission.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
gabarito_df = pd.read_csv(gabarito_path)

# Tratamento de dados ausentes
train_df['Age'].fillna(train_df['Age'].median(), inplace=True)
test_df['Age'].fillna(test_df['Age'].median(), inplace=True)
train_df['Embarked'].fillna(train_df['Embarked'].mode()[0], inplace=True)
test_df['Embarked'].fillna(test_df['Embarked'].mode()[0], inplace=True)
test_df['Fare'].fillna(test_df['Fare'].median(), inplace=True)

# Remover colunas irrelevantes
train_df.drop(columns=['Cabin', 'Ticket', 'Name', 'PassengerId'], inplace=True)
test_df.drop(columns=['Cabin', 'Ticket', 'Name', 'PassengerId'], inplace=True)

# Codificação de variáveis categóricas
train_df['Sex'] = train_df['Sex'].map({'female': 0, 'male': 1})
test_df['Sex'] = test_df['Sex'].map({'female': 0, 'male': 1})
train_df = pd.get_dummies(train_df, columns=['Embarked'], drop_first=True)
test_df = pd.get_dummies(test_df, columns=['Embarked'], drop_first=True)

# Separar variáveis preditoras e alvo
X = train_df.drop(columns=['Survived'])
y = train_df['Survived']

# Dividir dados em treino e validação
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Função para treinar e avaliar modelos
def evaluate_model(model, model_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return {
        'Acurácia': accuracy_score(y_val, y_pred),
        'Precisão': precision_score(y_val, y_pred),
        'Recall': recall_score(y_val, y_pred),
        'F1-Score': f1_score(y_val, y_pred)
    }

# Modelos
models = {
    "Naive Bayes": GaussianNB(),
    "Árvore de Decisão": DecisionTreeClassifier(random_state=42),
    "SVM": SVC(kernel="linear", random_state=42),
    "Rede Neural": MLPClassifier(hidden_layer_sizes=(10,), max_iter=1000, random_state=42)
}

# Avaliação dos modelos
results = {name: evaluate_model(model, name) for name, model in models.items()}
results_df = pd.DataFrame(results).T

# Treinar o melhor modelo (Árvore de Decisão) com todos os dados
best_model = DecisionTreeClassifier(random_state=42)
best_model.fit(X, y)

# Fazer previsões no conjunto de teste
y_test_pred = best_model.predict(test_df)

gabarito_survived = gabarito_df["Survived"].values

# Calcular métricas finais
final_results = {
    "Acurácia": accuracy_score(gabarito_survived, y_test_pred),
    "Precisão": precision_score(gabarito_survived, y_test_pred),
    "Recall": recall_score(gabarito_survived, y_test_pred),
    "F1-Score": f1_score(gabarito_survived, y_test_pred)
}
final_results_df = pd.DataFrame(final_results, index=["Melhor Modelo"])

# Exibir resultados
print("Resultados da Avaliação dos Modelos:\n", results_df)
print("\nResultados do Melhor Modelo no Conjunto de Teste:\n", final_results_df)

C:\Users\felpp\AppData\Local\Temp\ipykernel_38020\4019046700.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['Age'].fillna(train_df['Age'].median(), inplace=True)
C:\Users\felpp\AppData\Local\Temp\ipykernel_38020\4019046700.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as 

Resultados da Avaliação dos Modelos:
                    Acurácia  Precisão    Recall  F1-Score
Naive Bayes        0.782123  0.727273  0.695652  0.711111
Árvore de Decisão  0.821229  0.793651  0.724638  0.757576
SVM                0.776536  0.737705  0.652174  0.692308
Rede Neural        0.782123  0.734375  0.681159  0.706767

Resultados do Melhor Modelo no Conjunto de Teste:
                Acurácia  Precisão    Recall  F1-Score
Melhor Modelo  0.782297  0.672316  0.782895  0.723404
